# Analyse Att — reconstruction des opérations (v1, simplifié)

Deux étapes seulement, dans cet ordre, pour tous les tags Att du catalogue :

1. **Étape 1** — transformer le signal brut (Att, 1 point/minute) en une série simple `pas / repère / durée (min)` : un pas = un plateau de repère constant, mappé au numéro de pas du séquenceur via la table process (`OPxxxx_pas_reference.csv`).
2. **Étape 2** — reconstruire les opérations à partir de cette série : incrémenter un numéro d'opération (`operation_id`) séquentiellement, pas par pas, à chaque nouveau départ détecté (`att_analysis.reconstruct_operations`).

Pas d'exploration, pas de graphique, pas de synthèse SPC ici — juste les deux transformations de base, pour pouvoir vérifier pas à pas ce que fait l'algorithme avant d'aller plus loin.

In [1]:
import sys

import numpy as np
import pandas as pd
from IPython.display import display

from tools.OI_class_OP import OI_DataProcessor
import att_analysis as atta

pd.set_option('display.max_columns', None)

# att_tags_config lit documents/confidentiel/att_operations_config.json une
# seule fois, à l'import du module -- ce nettoyage force sa relecture sans
# avoir besoin de redémarrer le kernel après une modification du JSON.
for _mod in list(sys.modules):
    if _mod == 'att_tags_config' or _mod.startswith('att_tags_config.'):
        del sys.modules[_mod]

from att_tags_config import TAGS as tags, has_pas_reference, is_batch

processor = OI_DataProcessor(
    url_base='https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start='2026-01-01',
    end='2026-12-31',
    tags_selected=tags,
    tags_other=[],
    interval='PT01M',
    verbose=False,
    agg='FIRST',
)
processor.merge()
processor.data.describe()


,PU1410VA,PU1420VA,PU1430VA,PU1510VA,PU1520VA,PU1530VA,PU1540VA,PU1610VA,PU2810VA,PU2910VA,PU2310VA,PU2320VA,PU2340VA,PU2330VA,PU2410VA,PU2411VA,PU2420VA,PU2510VA,PU2520VA,PU3110VA,PU3120VA,PU3130VA,PU3210VA,PU3220VA,PU3230VA,PU3310VA,PU3320VA,PU3330VA
count,134138.000000,80621.000000,128795.000000,82468.000000,83419.000000,82495.000000,85918.000000,86422.000000,78809.000000,78616.000000,80937.000000,82472.000000,80630.000000,78644.000000,81498.000000,80442.000000,82985.000000,78469.000000,78651.000000,82987.000000,89834.000000,78816.000000,83597.000000,83439.000000,80802.000000,81770.000000,87628.000000,81324.000000
mean,1227.495229,484.462063,1178.360262,601.525622,592.148192,566.068974,710.120464,1051.305917,1245.573412,845.277170,804.959289,1214.664856,449.355451,673.309216,901.058308,633.868066,854.549497,872.127337,648.442741,1332.400328,1085.496638,630.716796,1123.304808,726.876760,521.177137,1001.383820,952.182647,908.480719
std,625.564278,297.854472,783.458593,359.485165,297.027362,201.405796,440.856889,454.246889,323.093101,287.806946,458.135552,723.636802,110.386735,200.672181,421.152555,184.299956,440.490972,243.572934,194.567119,740.714584,765.303846,69.688564,543.218351,334.473369,208.180411,633.327844,566.428874,240.425354
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1048.000000,230.000000,110.000000,320.000000,510.000000,510.000000,520.000000,1250.000000,1150.000000,960.000000,550.000000,450.000000,410.000000,750.000000,470.000000,620.000000,510.000000,950.000000,720.000000,620.000000,270.000000,620.000000,910.000000,510.000000,430.000000,410.000000,410.000000,930.000000
50%,1510.000000,450.000000,1720.000000,820.000000,520.000000,610.000000,610.000000,1250.000000,1410.000000,960.000000,720.000000,1310.000000,410.000000,750.000000,1110.000000,720.000000,810.000000,950.000000,720.000000,1810.000000,1050.000000,620.000000,1150.000000,630.000000,440.000000,1320.000000,1110.000000,930.000000
75%,1650.000000,760.000000,1720.000000,820.000000,910.000000,720.000000,1110.000000,1250.000000,1410.000000,960.000000,1260.000000,1950.000000,410.000000,750.000000,1250.000000,720.000000,1350.000000,950.000000,720.000000,2010.000000,1870.000000,620.000000,1620.000000,1120.000000,630.000000,1320.000000,1520.000000,930.000000
max,2190.000000,1460.000000,2740.000000,1050.000000,1230.000000,860.000000,1630.000000,1710.000000,1450.000000,1380.000000,1660.000000,2180.000000,740.000000,830.000000,1390.000000,750.000000,1450.000000,1330.000000,830.000000,2090.000000,2190.000000,720.000000,1760.000000,1420.000000,980.000000,2320.000000,2260.000000,1390.000000


## Étape 1 — série `pas / repère / durée (min)`

Pour chaque tag : nettoyage (`clean_dataframe`, exclut Att=0), extraction des plateaux (`extract_steps`), mappage au numéro de pas (`label_with_pas`). Résultat : une ligne par pas réellement traversé, avec sa durée en minutes.

In [2]:
pas_tables = {}

for tag_def in tags:
    label, nom = tag_def['tag'], tag_def['nom']
    if nom not in processor.data.columns or not has_pas_reference(tag_def):
        continue
    if not is_batch(tag_def):
        # Un PU en marche continue n'a pas de cycle discret pas1->pasN->
        # redémarrage : reconstruct_operations ne s'applique pas (cf.
        # att_tags_config.is_batch, documents/confidentiel/OPERATION.md).
        continue

    pas_reference = atta.load_pas_reference(tag_def['pas_reference'])
    cleaned = atta.clean_dataframe(processor.data[[nom]], value_col=nom)
    steps = atta.extract_steps(cleaned, value_col=nom)
    labeled = atta.label_with_pas(steps, pas_reference)

    pas_tables[label] = {'pas_reference': pas_reference, 'labeled': labeled}

recap_etape1 = pd.DataFrame([
    {
        'Tag': label,
        'N pas (total)': len(d['labeled']),
        'N pas non mappés': int(d['labeled']['pas_num'].isna().sum()),
    }
    for label, d in pas_tables.items()
])
recap_etape1


,Tag,N pas (total),N pas non mappés
0,PU1410VA_Att,25370,0
1,PU1420VA_Att,6052,0
2,PU1430VA_Att,17578,0
3,PU1510VA_Att,7281,0
4,PU1520VA_Att,7593,0
5,PU1530VA_Att,5904,0
6,PU1540VA_Att,10388,0
7,PU2810VA_Att,1514,0
8,PU2310VA_Att,6388,0
9,PU2320VA_Att,8879,0


In [3]:
# Changer cette valeur pour inspecter un autre tag.
example_label = 'PU1410VA_Att'

pas_tables[example_label]['labeled'][['pas_num', 'repere', 'start', 'duration_min']].head(30)

,pas_num,repere,start,duration_min
0,1.0,110,2026-01-01 00:00:00+00:00,207.0
1,2.0,230,2026-01-01 03:27:00+00:00,2.0
2,4.0,420,2026-01-01 03:29:00+00:00,4.0
3,6.0,650,2026-01-01 03:33:00+00:00,20.0
4,7.0,730,2026-01-01 03:53:00+00:00,46.0
5,7.0,740,2026-01-01 04:39:00+00:00,6.0
6,10.0,1030,2026-01-01 04:45:00+00:00,1.0
7,10.0,1035,2026-01-01 04:46:00+00:00,1.0
8,10.0,1040,2026-01-01 04:47:00+00:00,1.0
9,10.0,1045,2026-01-01 04:48:00+00:00,1.0


## Étape 2 — reconstruction séquentielle des opérations

`reconstruct_operations` parcourt la série de l'étape 1 pas par pas et incrémente `operation_id` à chaque nouveau départ détecté (retour au point d'entrée réel de l'opération, cf. `att_analysis.find_restart_reference` — pas de réglage manuel par opération, cf. `documents/confidentiel/OPERATION.md` section 7). Une régression qui n'est pas un nouveau départ est marquée `is_defaut`.

In [4]:
tag_operations = {}

for label, d in pas_tables.items():
    pas_reference, labeled = d['pas_reference'], d['labeled']

    restart_reference = atta.find_restart_reference(labeled, pas_reference)
    operations, steps = atta.reconstruct_operations(labeled, pas_reference, restart_reference=restart_reference)

    tag_operations[label] = {
        'pas_reference': pas_reference,
        'steps': steps,
        'operations': operations,
        'restart_reference': restart_reference,
    }

print(f"{len(tag_operations)} tags reconstruits")

21 tags reconstruits


In [5]:
# Meme tag que l'etape 1, avec operation_id/is_defaut ajoutes : la vue
# sequentielle pas par pas pour verifier visuellement ce que fait l'algo.
cols = ['start', 'pas_num', 'repere', 'duration_min', 'operation_id', 'is_defaut']
tag_operations[example_label]['steps'][cols].head(30)


,start,pas_num,repere,duration_min,operation_id,is_defaut
0,2026-01-01 00:00:00+00:00,1,110,207.0,0,False
1,2026-01-01 03:27:00+00:00,2,230,2.0,0,False
2,2026-01-01 03:29:00+00:00,4,420,4.0,0,False
3,2026-01-01 03:33:00+00:00,6,650,20.0,0,False
4,2026-01-01 03:53:00+00:00,7,730,46.0,0,False
5,2026-01-01 04:39:00+00:00,7,740,6.0,0,False
6,2026-01-01 04:45:00+00:00,10,1030,1.0,0,False
7,2026-01-01 04:46:00+00:00,10,1035,1.0,0,False
8,2026-01-01 04:47:00+00:00,10,1040,1.0,0,False
9,2026-01-01 04:48:00+00:00,10,1045,1.0,0,False


## Récapitulatif

In [6]:
# Temps de reference (BDP) : pour chaque tag, somme sur tous les pas reels
# du P10 (10e percentile) de duree observee pour ce pas -- cf.
# atta.reference_cycle_time. Independant de reconstruct_operations.
recap_rows = []

for label, res in tag_operations.items():
    ops = res['operations']
    tag_def = next(t for t in tags if t['tag'] == label)

    bdp = atta.reference_cycle_time(pas_tables[label]['labeled'], pas_tables[label]['pas_reference'])

    recap_rows.append({
        'Tag': label,
        'Opération': tag_def.get('operation', ''),
        'N opérations': len(ops),
        'N défauts (total)': int(ops['n_defauts'].sum()),
        'Durée totale — médiane (min)': round(ops['duration_min'].median(), 0),
        'Temps réf. — P10 (min)': round(bdp['total_min'], 0),
        'N pas réels sommés': bdp['n_pas_reels'],
        'Temps Réf. exploitant (min)': tag_def.get('temps_reference_min'),
    })

recap_table = pd.DataFrame(recap_rows)
recap_table['Écart au Réf. (%)'] = round(
    (recap_table['Temps réf. — P10 (min)'] - recap_table['Temps Réf. exploitant (min)'])
    / recap_table['Temps Réf. exploitant (min)'] * 100, 1
)
recap_table


,Tag,Opération,N opérations,N défauts (total),Durée totale — médiane (min),Temps réf. — P10 (min),N pas réels sommés,Temps Réf. exploitant (min),Écart au Réf. (%)
0,PU1410VA_Att,OP1410,415,88,523.0,103.0,16,425,-75.8
1,PU1420VA_Att,OP1420,414,0,522.0,176.0,9,222,-20.7
2,PU1430VA_Att,OP1430,139,56,1691.0,392.0,19,1027,-61.8
3,PU1510VA_Att,OP1510,469,115,480.0,32.0,8,412,-92.2
4,PU1520VA_Att,OP1520,470,0,477.0,77.0,12,387,-80.1
5,PU1530VA_Att,OP1530,469,0,477.0,88.0,7,418,-78.9
6,PU1540VA_Att,OP1540,469,0,477.0,154.0,16,391,-60.6
7,PU2810VA_Att,OP2810,84,0,3136.0,173.0,8,886,-80.5
8,PU2310VA_Att,OP2310,405,0,575.0,326.0,12,530,-38.5
9,PU2320VA_Att,OP2320,405,0,585.0,122.0,15,327,-62.7


In [7]:
# Detail pas par pas pour un tag donne (P10 et minimum brut, pour
# comparaison) -- utile pour voir quel pas tire "Temps ref. -- P10" vers le
# bas ou le haut.
detail_label = example_label
bdp_p10 = atta.reference_cycle_time(pas_tables[detail_label]['labeled'], pas_tables[detail_label]['pas_reference'])
bdp_min = atta.reference_cycle_time(
    pas_tables[detail_label]['labeled'], pas_tables[detail_label]['pas_reference'], percentile=0.0
)

detail = pd.DataFrame(bdp_p10['par_pas']).rename(columns={'duration_min': 'P10_duration_min'})
detail_min = pd.DataFrame(bdp_min['par_pas']).rename(columns={'duration_min': 'min_duration_min'})
detail = detail.merge(detail_min[['pas_num', 'min_duration_min']], on='pas_num', how='left')
detail.sort_values('P10_duration_min')


,pas_num,P10_duration_min,code_court,min_duration_min
2,4.0,1.0,TEST ETANCH.,1.0
3,5.0,1.0,DEB.CH.AIP,1.0
6,10.0,1.0,CHARGE H2SO4,1.0
7,11.0,1.0,CHAUFF.ACID.,1.0
8,12.0,1.0,CHARGE PAL,1.0
11,15.0,1.0,CHAUF.REFLUX,1.0
12,16.0,1.0,DISTI.ACET.,1.0
1,2.0,2.0,INIT.OPER.,1.0
14,20.0,2.0,REFROIDISS.,1.0
9,13.0,4.0,MARCHE GAV,2.0
